# RNN architecture and forward propagation
Three tiny movie reviews, one-hot encoded. We count the parameters of a SimpleRNN, then run forward
propagation by hand with NumPy and check that Keras gives the same numbers.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## One-hot encoding the reviews
Vocabulary of 5 words; each word becomes a vector of 5 numbers with a single 1.

In [ ]:
vocab = ["movie", "was", "good", "bad", "not"]
reviews = ["movie was good", "movie was bad", "movie was not good"]
sentiment = [1, 0, 0]
one_hot = {w: np.eye(5)[i] for i, w in enumerate(vocab)}
X = [np.array([one_hot[w] for w in r.split()]) for r in reviews]
for r, x in zip(reviews, X):
    print(f"{r!r:22} shape {x.shape}  (time steps, input features)")

In [ ]:
# Keras wants one 3D tensor (batch, time steps, input features): pad the short reviews with zero vectors
batch = keras.utils.pad_sequences(X, dtype="float32", padding="post")
print("batch shape:", batch.shape)

## The architecture: 5 inputs, 3 recurrent nodes, 1 output node

In [ ]:
model = keras.Sequential([keras.Input(shape=(4, 5)),
                          keras.layers.SimpleRNN(3),
                          keras.layers.Dense(1, activation="sigmoid")])
model.summary()
for w, name in zip(model.get_weights(), ["W_i (kernel)", "W_h (recurrent_kernel)", "b_h", "W_o", "b_o"]):
    print(f"{name:24} shape {w.shape}")
assert model.count_params() == 31

## Forward propagation with chosen weights
Simple weights so the numbers can be followed by hand. Biases are 0.

In [ ]:
W_i = np.array([[0.2, -0.1, 0.0],    # movie
                [0.0, 0.1, 0.1],     # was
                [0.8, 0.3, -0.5],    # good
                [-0.8, -0.3, 0.5],   # bad
                [-0.6, 0.2, 0.4]])   # not
W_h = np.array([[0.5, 0.0, 0.1],
                [0.2, 0.4, 0.0],
                [0.0, -0.3, 0.5]])
b_h = np.zeros(3)
W_o = np.array([[1.5], [0.5], [-1.0]])
b_o = np.zeros(1)


def forward(x, W_h=W_h):
    """Return every hidden state h_1..h_T and the prediction."""
    h, states = np.zeros(3), []                     # h_0 = zeros
    for x_t in x:
        h = np.tanh(x_t @ W_i + h @ W_h + b_h)
        states.append(h)
    y_hat = 1 / (1 + np.exp(-(h @ W_o + b_o)))     # sigmoid on the last hidden state
    return np.array(states), y_hat.item()


states, y_hat = forward(X[0])
for t, (w, h) in enumerate(zip(reviews[0].split(), states), start=1):
    pre = (X[0][t - 1] @ W_i + (states[t - 2] if t > 1 else np.zeros(3)) @ W_h)
    print(f"t={t} {w:6} x_t W_i + h_(t-1) W_h = {np.round(pre, 3)}   h_t = {np.round(h, 3)}")
print("h_3 W_o =", np.round(states[-1] @ W_o, 3), "  y_hat =", round(y_hat, 3))

In [ ]:
# Same weights in Keras: the outputs must match
model.set_weights([W_i, W_h, b_h, W_o, b_o])
keras_rnn = keras.Sequential([keras.Input(shape=(None, 5)), keras.layers.SimpleRNN(3, return_sequences=True)])
keras_rnn.set_weights([W_i, W_h, b_h])
keras_states = keras_rnn.predict(X[0][None], verbose=0)[0]
keras_y = model.predict(X[0][None, :, :].astype("float32"), verbose=0).item()
print("Keras hidden states:\n", np.round(keras_states, 3))
print("Keras prediction:", round(keras_y, 3))
assert np.allclose(keras_states, states, atol=1e-6) and abs(keras_y - y_hat) < 1e-6

In [ ]:
# Save every step for the animation and the Note
rows = [dict(t=t, word=w, h1=h[0], h2=h[1], h3=h[2]) for t, (w, h) in enumerate(zip(reviews[0].split(), states), 1)]
pd.DataFrame(rows).assign(y_hat=y_hat).round(4).to_csv(DATA / "worked_example.csv", index=False)

## The hidden state carries earlier words forward
Two reviews that differ only in the middle word. With the recurrent weights W_h, the last hidden state
differs; with W_h set to 0, the last hidden state sees only the last word, so it is identical.

In [ ]:
a, b = [np.array([one_hot[w] for w in s.split()]) for s in ("movie was good", "movie not good")]
for label, Wh in (("with W_h", W_h), ("W_h = 0", np.zeros((3, 3)))):
    ha, ya = forward(a, Wh)
    hb, yb = forward(b, Wh)
    print(f"{label:9} h_3 'movie was good' = {np.round(ha[-1], 3)}  'movie not good' = {np.round(hb[-1], 3)}"
          f"  y_hat {ya:.3f} vs {yb:.3f}")